# 토큰화: BPE와 SentencePiece - BPE 토큰화 알고리즘

- Tutorial ID: `adv-9-1`
- Tutorial: 토큰화: BPE와 SentencePiece
- Section ID: `adv-9-1-1`
- Section: BPE 토큰화 알고리즘


In [ ]:
# ============================================================
# 코드 읽는 법 — BPE 토큰화 알고리즘
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 입력 데이터가 어떤 중간 변수들을 거쳐 최종 출력으로 변환되는지 shape 중심으로 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("BPE (Byte-Pair Encoding) 토큰화")
print("=" * 60)

In [ ]:
# 1. BPE 알고리즘 구현

In [ ]:
print("\n1. BPE 알고리즘 시뮬레이션")
print("-" * 45)

def get_pair_counts(tokens_list):
    """인접 토큰 쌍의 빈도 계산"""
    counts = {}
        for tokens in tokens_list:
                for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i+1])
            counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge_pair(tokens_list, pair, new_token):
    """가장 빈번한 쌍을 새 토큰으로 병합"""
        result = []
        for tokens in tokens_list:
        new_tokens = []
        i = 0
                while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i+1] == pair[1]:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        result.append(new_tokens)
    return result

# 코퍼스
corpus = ["low", "lower", "newest", "widest", "low", "low", "lower"]
print(f"코퍼스: {corpus}")

# 문자 단위로 분할
tokens_list = [[c for c in word] + ['</w>'] for word in corpus]
print(f"초기 토큰: {tokens_list[0]}")

# 초기 어휘
vocab = set()
for tokens in tokens_list:
    vocab.update(tokens)
print(f"초기 어휘 크기: {len(vocab)}")
print(f"초기 어휘: {sorted(vocab)}")

# BPE 반복
for step in range(8):
    pair_counts = get_pair_counts(tokens_list)
    if not pair_counts:
        break
    
    best_pair = max(pair_counts, key=pair_counts.get)
    new_token = best_pair[0] + best_pair[1]
    
    print(f"\n단계 {step+1}: '{best_pair[0]}'+'{best_pair[1]}' → '{new_token}' (빈도: {pair_counts[best_pair]})")
    
    tokens_list = merge_pair(tokens_list, best_pair, new_token)
    vocab.add(new_token)
    
    print(f"  어휘 크기: {len(vocab)}")
        for i, (word, tokens) in enumerate(zip(corpus[:3], tokens_list[:3])):
        print(f"  '{word}' → {tokens}")

print(f"\n최종 어휘: {sorted(vocab)}")

In [ ]:
# 2. 토큰화의 영향

In [ ]:
print("\n2. 토큰화가 트랜스포머에 미치는 영향")
print("-" * 45)
print("""
어휘 크기 선택:
  작은 어휘 (1000): 시퀀스 길이 ↑, 임베딩 작음
  큰 어휘 (100000): 시퀀스 길이 ↓, 임베딩 큼

실제 모델 어휘 크기:
  GPT-2:     50,257 (BPE)
  LLaMA:     32,000 (SentencePiece)
  GPT-4:    100,256 (tiktoken)
  
서브워드 토큰화의 장점:
  1. OOV(미등록 단어) 없음
  2. 형태소 정보 보존 ('un' + 'happy')
  3. 다국어 지원 가능
  
주의: 토큰 경계가 의미 경계와 불일치할 수 있음!
  예: " New" + " York" vs "New" + "York"
""")